# 02 — NLP Data Preprocessing & EDA (Flood Control: YouTube + Rappler)

**you need:**
- `cleaned_corpus.xlsx` (sheet: `cleaned_corpus`) — the combined table in step 1
- `Basic Stopwords.txt` — common English + Filipino words needed to be filtered out
- `Domain-Specific Stopwords.txt` — all those flood/control/scandal words that show up a lot of times, plus entities and random platform stuff from youtube and rappler

**this notebook will create:**
- `processed/clean_tokens.parquet` — all rows broken down into tokens with stopwords removed
- `processed/unigram_freq.csv`, `processed/bigram_freq.csv` — basically word counts for the whole dataset
- `processed/unigram_freq_by_platform.csv` — same thing but split by platform
- Some graphs in the `figures/` folder like top words, how long documents are, etc.

In [ ]:
from pathlib import Path
import os

BASE = Path("..")
DATA_DIR = BASE / "data"
RAW = DATA_DIR / "raw"
PROC = DATA_DIR / "processed"
FIG = BASE / "figures"

for p in (RAW, PROC, FIG):
    os.makedirs(p, exist_ok=True)

## Setup

In [ ]:

# install these ok
# !pip install pandas numpy matplotlib openpyxl scikit-learn emoji regex pyarrow
import os, re, math, string
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
from emoji import replace_emoji

## Load cleaned corpus

In [ ]:
import pandas as pd
from pathlib import Path

# this function uses the 'RAW' var defined in the setup cell
def load_first_available():
    csv = RAW / "cleaned_corpus.csv"
    xls = RAW / "cleaned_corpus.xlsx"
    if csv.exists():
        print(f"Loading from: {csv}")
        return pd.read_csv(csv)
    elif xls.exists():
        print(f"Loading from: {xls}")
        xf = pd.ExcelFile(xls)
        SHEET = "cleaned_corpus" if "cleaned_corpus" in xf.sheet_names else xf.sheet_names[0]
        return pd.read_excel(xls, sheet_name=SHEET)
    else:
        raise FileNotFoundError("cleaned_corpus.csv/xlsx not found in data/raw")

df = load_first_available()
print("Loaded rows:", len(df), "columns:", list(df.columns))
df.head(2)

## Stopwords (basic + domain)

In [ ]:
import re
from pathlib import Path

def load_list(path: Path):
    if not path.exists(): return []
    with open(path, "r", encoding="utf-8") as f:
        return [ln.strip().lower() for ln in f if ln.strip() and not ln.strip().startswith("#")]

def load_domain_stopwords(path: Path):
    single_words, multi_phrases = [], []
    if not path.exists():
        return [], []

    for line in path.read_text(encoding="utf-8").splitlines():
        t = line.strip().lower()
        if not t or t.startswith("#"):   # allow comments like "# --- MULTI-WORD ..."
            continue
        if " " in t:
            multi_phrases.append(t)
        else:
            single_words.append(t)

    single_words = sorted(list(set(single_words)))
    # sort phrases by length desc so longer ones match first
    multi_phrases = sorted(list(set(multi_phrases)), key=len, reverse=True)

    print(f"Loaded {len(single_words)} single-word and {len(multi_phrases)} multi-word domain stopwords.")
    return single_words, multi_phrases

# Load all stopword lists
CONFIG_PATH = BASE / "configs"

# Load Basic and Tagalog
basic_sw = set(load_list(CONFIG_PATH / "Basic Stopwords.txt"))
tagalog_sw = set(load_list(CONFIG_PATH / "tagalog_stopwords.txt"))

# Load Domain which includes both single and multi-word
DOMAIN_SW, DOMAIN_PHRASES = load_domain_stopwords(CONFIG_PATH / "Domain-Specific Stopwords.txt")

# Create the final set of SINGLE stopwords
STOP = basic_sw | tagalog_sw | set(DOMAIN_SW)
print(f"Total single-word stopwords loaded: {len(STOP)}")

# DOMAIN_PHRASES becomes a separate list
print(f"Total multi-word phrases to remove: {len(DOMAIN_PHRASES)}")

## Clean, normalize, and tokenize

In [ ]:
import emoji
import re
from collections import Counter

# Multi-word phrase removal
def remove_multiword_phrases(text: str, phrases: list[str]) -> str:
    s = text
    s = s.replace("\u2019", "'").lower()
    for p in phrases:
        pat = r"\b" + re.escape(p) + r"\b"
        s = re.sub(pat, " ", s, flags=re.IGNORECASE)
    # collapse extra spaces
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Single-word cleaning pipeline

PUNCT = r"[^0-9a-zA-Záéíóúüñçâêîôûäëïöüß₱\s]"
WS = re.compile(r"\s+")

import re
def is_numberlike(t):
    # Checks for integers or year-like tokens
    if t.isdigit():
        return True
    # Checks for timestamps or dates
    if bool(re.fullmatch(r"(\d+[:/\.]\d+)+", t)):
        return True
    return False

def basic_clean(s: str) -> str:
    if s is None:
        return ""
    s = str(s)

    # Remove URLs and Emojis
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = emoji.replace_emoji(s, replace=" ")

    # Remove multi-word phrases before punctuation stripping
    s = remove_multiword_phrases(s, DOMAIN_PHRASES)

    # de-noise punctuation after phrases are gone
    s = s.lower()
    s = re.sub(PUNCT, " ", s)
    s = WS.sub(" ", s).strip()
    return s

def tokenize_and_stop(s: str) -> list[str]:
    return [t for t in s.split() 
            if t not in STOP 
            and len(t) > 2 
            and not is_numberlike(t)]

test_text = "This is AI generated summarization, please refer full article. Also, hello po."
print(f"Original: {test_text}")

clean_version = basic_clean(test_text)
print(f"Cleaned:  {clean_version}")

tokens_version = tokenize_and_stop(clean_version)
print(f"Tokens:   {tokens_version}")

df["text_norm"] = df["text"].map(basic_clean)
df["tokens_nostop"] = df["text_norm"].map(tokenize_and_stop)
df["tok_ns_count"] = df["tokens_nostop"].map(len)

# for reference (count before stopword removal)
df["tok_count"] = df["text_norm"].map(lambda s: len(s.split())) 

print("\nNull texts:", df["text"].isna().sum())
print("Sample tokens:", df["tokens_nostop"].head(2).tolist())

## Save token table

In [ ]:

# this list now correctly includes 'text_norm' and the updated tok_count
out_cols = ['title','link','date_published','source','tok_count','tok_ns_count','tokens_nostop','text_norm']

final_out_cols = [col for col in out_cols if col in df.columns]
df_out = df[final_out_cols].copy()

parquet_path = PROC / 'clean_tokens.parquet'

df_out.to_parquet(parquet_path, index=False)

print(f"Saved: {parquet_path}")
print(f"Total rows saved: {len(df_out)}")

## Build unigram or bigram frequencies

In [ ]:
#  global unigrams & bigrams
def flatten(iterables):
    for it in iterables:
        for x in it:
            yield x

unigrams = list(flatten(df['tokens_nostop']))
uni_counts = Counter(unigrams)
uni_df = pd.DataFrame(uni_counts.most_common(), columns=['term','freq'])

# save global unigram frequency
uni_df.to_csv(PROC / 'unigram_freq.csv', index=False)

# per platform unigram frequency
uni_by_platform = (
    df.explode('tokens_nostop')
      .dropna(subset=['tokens_nostop'])
      .groupby(['source','tokens_nostop'])
      .size().reset_index(name='freq')
      .rename(columns={'tokens_nostop':'term'})
)
uni_by_platform.to_csv(PROC / 'unigram_freq_by_platform.csv', index=False)

# Bigrams using sklearn
cv = CountVectorizer(ngram_range=(2,2), min_df=5, max_df=0.9, tokenizer=lambda s: s.split(), preprocessor=lambda s: s)
joined = df['tokens_nostop'].map(lambda toks: " ".join(toks))
X = cv.fit_transform(joined)
bg_counts = np.asarray(X.sum(axis=0)).ravel().tolist()
bg_df = pd.DataFrame({'term': cv.get_feature_names_out(), 'freq': bg_counts}).sort_values('freq', ascending=False)
bg_df.to_csv(PROC / 'bigram_freq.csv', index=False)

uni_df.head(10), bg_df.head(10)

## Quick EDA plots

In [ ]:
# Quick EDA figures
# Top 30 unigrams
topn = 30
top_uni = uni_df.head(topn)
plt.figure(figsize=(10,6))
plt.barh(top_uni['term'][::-1], top_uni['freq'][::-1])
plt.title('Top Unigrams (No Stopwords)')
plt.tight_layout()
plt.savefig(FIG / 'top_unigrams.png', dpi=160)
plt.close()

# document length histograms per platform
for src, sub in df.groupby('source'):
    plt.figure(figsize=(8,5))
    sub['tok_ns_count'].clip(upper=sub['tok_ns_count'].quantile(0.99)).hist(bins=40)
    plt.title(f'Doc Length (no-stop) – {src}')
    plt.xlabel('tokens')
    plt.ylabel('docs')
    plt.tight_layout()
    plt.savefig(FIG / f'len_hist_{src}.png', dpi=160)
    plt.close()

# share of rows per platform
plt.figure(figsize=(6,6))
df['source'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title(f'Rows by Platform in Cleaned Corpus (n={len(df)})')
plt.ylabel('')
plt.tight_layout()
plt.savefig(FIG / 'platform_share.png', dpi=160)
plt.close()

from wordcloud import WordCloud

# word cloud from unigram frequencies, which was created in the previous cell
word_freqs = dict(zip(uni_df['term'], uni_df['freq']))

wc = WordCloud(width=800, 
               height=400, 
               background_color='white', 
               colormap='viridis').generate_from_frequencies(word_freqs)

plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Top Unigrams (Stopwords Removed)')
plt.tight_layout()
plt.savefig(FIG / 'word_cloud_unigrams.png', dpi=160)
plt.close()

import seaborn as sns
import matplotlib.pyplot as plt

# sort by platform then frequency in descending order
uni_by_platform_sorted = (
    uni_by_platform
    .sort_values(['source', 'freq'], ascending=[True, False])
)

# top Unigrams by Platform which is now correctly sorted
g = sns.catplot(
    data=uni_by_platform_sorted.groupby('source').head(25),
    y='term',
    x='freq',
    col='source',
    kind='bar',
    sharex=False,
    sharey=False,
    height=8,
    aspect=0.7
)
g.fig.suptitle('Top 25 Unigrams by Platform', y=1.03)
g.set_axis_labels('Frequency', 'Term')
g.set_titles(col_template="{col_name}") # titles will be 'rappler' and 'youtube'
plt.tight_layout()
plt.savefig(FIG / 'top_unigrams_by_platform.png', dpi=160)
plt.close()

print("Saved corrected 'top_unigrams_by_platform.png'")

print(os.listdir(FIG))